# V7 Inference - DINOv2-L + EVA-02-L Cross-Run Ensemble

**Lade Checkpoints aus zwei verschiedenen Trainings-Runs:**
- DINOv2 Large @ 518 -> `runs/v5_t2/best_dinov2l_fold_*.pth`
- EVA-02  Large @ 448 -> `runs/v6_t2/best_eva02l_fold_*.pth`

**ConvNeXt wird ignoriert** (in V5/V6 zu schwach: 0.76 vs 0.88-0.90 der ViTs).

**Pipeline:**
1. Alle Modelle einmal mit 6-fach TTA inferenzieren
2. Probs sammeln
3. Drei Submissions schreiben (verschiedene DINOv2/EVA-Gewichte) zum Vergleich

## Config

In [1]:
import os

TAG = "T2"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"

_candidates = [
    "multiview_pig_posture_recognition",
    "./multiview_pig_posture_recognition",
    "/datasets/multi-view-pig-posture-recognition",
    "/multi-view-pig-posture-recognition",
]
DATA_ROOT = None
for _p in _candidates:
    if os.path.isdir(_p):
        DATA_ROOT = _p
        break
assert DATA_ROOT is not None
print(f"DATA_ROOT = {os.path.abspath(DATA_ROOT)}")

TEST_CSV  = os.path.join(DATA_ROOT, "test.csv")
IMG_DIR   = os.path.join(DATA_ROOT, "test_images")

# --- Quellen pro Architektur ---
# Tuple: (prefix, ckpt_dir, infer_img_size)
ARCH_SOURCES = [
    ("dinov2l", f"runs/v5_{TAG.lower()}", 518),
    ("eva02l",  f"runs/v6_{TAG.lower()}", 448),
]

# Gefilterte Checkpoint-Liste sammeln
CKPT_PATHS = []
for prefix, ckpt_dir, _sz in ARCH_SOURCES:
    if not os.path.isdir(ckpt_dir):
        print(f"  WARN: {ckpt_dir} existiert nicht, ueberspringe {prefix}")
        continue
    for f in sorted(os.listdir(ckpt_dir)):
        if f.startswith(f"best_{prefix}_fold_") and f.endswith(".pth"):
            CKPT_PATHS.append(os.path.join(ckpt_dir, f))

INFER_IMG_SIZE = {prefix: sz for prefix, _d, sz in ARCH_SOURCES}

BATCH_SIZE   = 32
NUM_WORKERS  = 16
USE_TTA      = True
PAD_RATIO    = 0.1
NUM_CLASSES  = 5

CLASS_NAMES = ["Lateral_lying_left", "Lateral_lying_right",
               "Sitting", "Standing", "Sternal_lying"]

ADAPT_BN         = True   # ViTs haben keine BN -> wird auto uebersprungen
BN_ADAPT_BATCHES = 50

# --- Drei Varianten zum Vergleich ---
# (name, weights-dict)
VARIANTS = [
    ("A_equal",         {"dinov2l": 1.0, "eva02l": 1.0}),
    ("B_dinov2_strong", {"dinov2l": 1.5, "eva02l": 1.0}),  # DINOv2 dominiert leicht (war 0.90 CV)
    ("C_eva_strong",    {"dinov2l": 1.0, "eva02l": 1.5}),  # falls EVA-02 besser generalisiert
]

# --- Pseudo-Label Export (default aus, aktiviere wenn V8 retrain geplant) ---
EXPORT_PSEUDO_LABELS   = True
PSEUDO_TOP_K_PER_CLASS = 200
PSEUDO_MIN_CONFIDENCE  = 0.60

print(f"Tag: {TAG}  |  Checkpoints gesamt: {len(CKPT_PATHS)}")
for p in CKPT_PATHS:
    print(f"  - {p}")
print(f"\nVarianten: {len(VARIANTS)}")
for name, w in VARIANTS:
    print(f"  {name}: weights={w}")

DATA_ROOT = /datasets/multi-view-pig-posture-recognition
Tag: T2  |  Checkpoints gesamt: 6
  - runs/v5_t2/best_dinov2l_fold_1.pth
  - runs/v5_t2/best_dinov2l_fold_2.pth
  - runs/v5_t2/best_dinov2l_fold_3.pth
  - runs/v6_t2/best_eva02l_fold_1.pth
  - runs/v6_t2/best_eva02l_fold_2.pth
  - runs/v6_t2/best_eva02l_fold_3.pth

Varianten: 3
  A_equal: weights={'dinov2l': 1.0, 'eva02l': 1.0}
  B_dinov2_strong: weights={'dinov2l': 1.5, 'eva02l': 1.0}
  C_eva_strong: weights={'dinov2l': 1.0, 'eva02l': 1.5}


## Imports

In [2]:
import os, ast, re
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
import torchvision.transforms as T
import timm

import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

<jemalloc>: Unsupported system page size


Device: cuda


## Dataset + TTA + Loader Helpers

In [3]:
class PigTestDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, pad_ratio=0.1):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.pad_ratio = pad_ratio

    def __len__(self): return len(self.df)

    def _crop(self, img, bbox):
        W, H = img.size
        x, y, w, h = [float(v) for v in ast.literal_eval(bbox)]
        px, py = w * self.pad_ratio, h * self.pad_ratio
        x1 = max(0, int(x - px));  y1 = max(0, int(y - py))
        x2 = min(W, int(x+w+px));  y2 = min(H, int(y+h+py))
        return img.crop((x1, y1, x2, y2))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row["image_id"])).convert("RGB")
        crop = self._crop(img, row["bbox"])
        if self.transform:
            crop = self.transform(crop)
        return crop, row["row_id"]


NORM = [[0.485, 0.456, 0.406], [0.229, 0.224, 0.225]]

def build_tta_configs(img_size):
    S = img_size
    return [
        (T.Compose([T.Resize((S, S), interpolation=T.InterpolationMode.BICUBIC),
                    T.ToTensor(), T.Normalize(*NORM)]), False),
        (T.Compose([T.Resize((S, S), interpolation=T.InterpolationMode.BICUBIC),
                    T.RandomHorizontalFlip(p=1.0),
                    T.ToTensor(), T.Normalize(*NORM)]), True),
        (T.Compose([T.Resize((S+32, S+32), interpolation=T.InterpolationMode.BICUBIC),
                    T.CenterCrop(S),
                    T.ToTensor(), T.Normalize(*NORM)]), False),
        (T.Compose([T.Resize((S+32, S+32), interpolation=T.InterpolationMode.BICUBIC),
                    T.CenterCrop(S), T.RandomHorizontalFlip(p=1.0),
                    T.ToTensor(), T.Normalize(*NORM)]), True),
        (T.Compose([T.Resize((S+64, S+64), interpolation=T.InterpolationMode.BICUBIC),
                    T.CenterCrop(S),
                    T.ToTensor(), T.Normalize(*NORM)]), False),
        (T.Compose([T.Resize((int(S*0.75), int(S*0.75))),
                    T.Resize((S, S), interpolation=T.InterpolationMode.BICUBIC),
                    T.ToTensor(), T.Normalize(*NORM)]), False),
    ]


def adapt_batch_norm(model, loader, device, n_batches=50):
    bn_layers = [m for m in model.modules()
                 if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.SyncBatchNorm))]
    if not bn_layers:
        return
    for bn in bn_layers:
        bn.running_mean.zero_()
        bn.running_var.fill_(1)
        bn.momentum = None
    model.train()
    with torch.no_grad():
        for i, (imgs, _) in enumerate(loader):
            if i >= n_batches: break
            model(imgs.to(device))
    model.eval()


def infer_prefix(ckpt_path):
    name = os.path.basename(ckpt_path)
    m = re.match(r"best_([a-zA-Z0-9]+)_fold_\d+\.pth", name)
    return m.group(1) if m else "unknown"


def fold_num(ckpt_path):
    m = re.search(r"fold_(\d+)\.pth$", ckpt_path)
    return int(m.group(1)) if m else -1


def load_vit_with_resampling(name, ckpt, num_classes, infer_size):
    model = timm.create_model(name, pretrained=False, num_classes=num_classes, img_size=infer_size)
    state_dict = dict(ckpt["model"])
    if "pos_embed" in state_dict:
        old_pe = state_dict["pos_embed"]
        if old_pe.shape != model.pos_embed.shape:
            try:
                from timm.layers import resample_abs_pos_embed
            except ImportError:
                from timm.models.layers import resample_abs_pos_embed
            num_prefix = getattr(model, "num_prefix_tokens", 1)
            print(f"    Interpoliere pos_embed: {tuple(old_pe.shape)} -> {tuple(model.pos_embed.shape)}")
            state_dict["pos_embed"] = resample_abs_pos_embed(
                old_pe, new_size=model.patch_embed.grid_size, num_prefix_tokens=num_prefix
            )
    model.load_state_dict(state_dict, strict=False)
    return model


@torch.no_grad()
def predict_tta(model, df, img_dir, tta_configs, batch_size):
    all_probs = []
    for i, (tf, is_flipped) in enumerate(tta_configs):
        ds = PigTestDataset(df, img_dir, transform=tf, pad_ratio=PAD_RATIO)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
        probs = []
        for imgs, _ in tqdm(loader, desc=f"  TTA {i+1}/{len(tta_configs)}", leave=False):
            with autocast():
                logits = model(imgs.to(DEVICE))
            p = F.softmax(logits, dim=1).cpu().numpy()
            if is_flipped:
                p[:, [0, 1]] = p[:, [1, 0]]
            probs.append(p)
        all_probs.append(np.vstack(probs))
    return np.mean(all_probs, axis=0)

print("Helpers geladen.")

Helpers geladen.


## Alle Modelle EINMAL inferenzen und Probs speichern

6 Modelle (3 DINOv2-L + 3 EVA-02-L) * 6 TTAs = teurer Schritt. Danach
Variantenkombination in Sekunden.

In [4]:
test_df = pd.read_csv(TEST_CSV)
print(f"Test-Instanzen: {len(test_df)}")

model_probs = []  # list of dicts

for idx, path in enumerate(CKPT_PATHS):
    prefix = infer_prefix(path)
    fnum = fold_num(path)
    print(f"\nModell {idx+1}/{len(CKPT_PATHS)}: {os.path.basename(path)} (arch={prefix}, fold={fnum})")

    ckpt = torch.load(path, map_location="cpu")
    name = ckpt.get("model_name", "unknown")
    infer_size = INFER_IMG_SIZE.get(prefix, 384)

    print(f"  {name}  |  Val F1: {ckpt.get('val_f1', 0):.4f}  |  Infer@{infer_size}px")

    try:
        model = load_vit_with_resampling(name, ckpt, NUM_CLASSES, infer_size)
    except TypeError:
        model = timm.create_model(name, pretrained=False, num_classes=NUM_CLASSES)
        model.load_state_dict(ckpt["model"])

    model.to(DEVICE).eval()

    if ADAPT_BN:
        bn_tf = T.Compose([
            T.Resize((infer_size, infer_size), interpolation=T.InterpolationMode.BICUBIC),
            T.ToTensor(), T.Normalize(*NORM),
        ])
        bn_ds = PigTestDataset(test_df, IMG_DIR, transform=bn_tf, pad_ratio=PAD_RATIO)
        bn_loader = DataLoader(bn_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True)
        adapt_batch_norm(model, bn_loader, DEVICE, n_batches=BN_ADAPT_BATCHES)

    tta_cfg = build_tta_configs(infer_size) if USE_TTA else [build_tta_configs(infer_size)[0]]
    probs = predict_tta(model, test_df, IMG_DIR, tta_cfg, BATCH_SIZE)

    model_probs.append({
        "prefix": prefix,
        "fold": fnum,
        "probs": probs,
        "val_f1": ckpt.get("val_f1", 0),
        "path": path,
    })

    del model
    import gc; gc.collect()
    torch.cuda.empty_cache()

print(f"\n{len(model_probs)} Modelle inferenziert.")

Test-Instanzen: 11708

Modell 1/6: best_dinov2l_fold_1.pth (arch=dinov2l, fold=1)
  vit_large_patch14_dinov2.lvd142m  |  Val F1: 0.8768  |  Infer@518px


  TTA 1/6:   0%|          | 0/366 [00:00<?, ?it/s]

  TTA 2/6:   0%|          | 0/366 [00:00<?, ?it/s]

  TTA 3/6:   0%|          | 0/366 [00:00<?, ?it/s]

  TTA 4/6:   0%|          | 0/366 [00:00<?, ?it/s]

...

  TTA 3/6:   0%|          | 0/366 [00:00<?, ?it/s]

  TTA 4/6:   0%|          | 0/366 [00:00<?, ?it/s]

  TTA 5/6:   0%|          | 0/366 [00:00<?, ?it/s]

  TTA 6/6:   0%|          | 0/366 [00:00<?, ?it/s]


6 Modelle inferenziert.


## Varianten kombinieren + Submissions schreiben

In [5]:
results = []

# Letzte final_probs fuer Pseudo-Label-Export merken (aus Variante A_equal)
last_final_probs = None

for variant_name, weights in VARIANTS:
    print(f"\n{'='*60}")
    print(f"  Variante: {variant_name}")
    print(f"{'='*60}")
    print(f"  Weights: {weights}")

    selected = []
    total_weight_sum = 0.0
    for m in model_probs:
        w = weights.get(m["prefix"], 0.0)
        if w <= 0:
            continue
        selected.append((m, w))
        total_weight_sum += w

    if not selected:
        print(f"  KEINE Modelle nach Filter!")
        continue

    print(f"  Aktive Modelle ({len(selected)}):")
    for m, w in selected:
        print(f"    - {os.path.basename(m['path'])} (f1={m['val_f1']:.4f})  weight={w}")

    weighted_probs = np.zeros_like(selected[0][0]["probs"])
    for m, w in selected:
        weighted_probs += m["probs"] * w
    final_probs = weighted_probs / total_weight_sum
    predictions = final_probs.argmax(axis=1)

    if variant_name == "A_equal":
        last_final_probs = final_probs

    out_file = f"{TAG}_v7_{variant_name}.csv"
    submission = pd.DataFrame({
        "row_id": test_df["row_id"].values,
        "class_id": predictions.astype(int),
    })
    submission.to_csv(out_file, index=False)

    print(f"\n  Submission: {out_file}")
    print(f"  Verteilung:")
    for c in range(NUM_CLASSES):
        cnt = (submission["class_id"] == c).sum()
        pct = 100 * cnt / len(submission)
        bar = "#" * int(30 * cnt / len(submission))
        print(f"    {c} - {CLASS_NAMES[c]:<22} {bar:<30} {cnt:>5} ({pct:.1f}%)")

    results.append({
        "name": variant_name,
        "file": out_file,
        "n_models": len(selected),
    })

print(f"\n\n{'='*60}")
print(f"  UEBERSICHT")
print(f"{'='*60}")
for r in results:
    print(f"  {r['name']:<25} -> {r['file']}  ({r['n_models']} Modelle)")


  Variante: A_equal
  Weights: {'dinov2l': 1.0, 'eva02l': 1.0}
  Aktive Modelle (6):
    - best_dinov2l_fold_1.pth (f1=0.8768)  weight=1.0
    - best_dinov2l_fold_2.pth (f1=0.9560)  weight=1.0
    - best_dinov2l_fold_3.pth (f1=0.8678)  weight=1.0
    - best_eva02l_fold_1.pth (f1=0.8722)  weight=1.0
    - best_eva02l_fold_2.pth (f1=0.9209)  weight=1.0
    - best_eva02l_fold_3.pth (f1=0.8485)  weight=1.0

  Submission: T2_v7_A_equal.csv
  Verteilung:
    0 - Lateral_lying_left     ###                             1327 (11.3%)
    1 - Lateral_lying_right    ###                             1493 (12.8%)
    2 - Sitting                #                                412 (3.5%)
    3 - Standing               ###############                 5911 (50.5%)
    4 - Sternal_lying          ######                          2565 (21.9%)

  Variante: B_dinov2_strong
  Weights: {'dinov2l': 1.5, 'eva02l': 1.0}
  Aktive Modelle (6):
    - best_dinov2l_fold_1.pth (f1=0.8768)  weight=1.5
    - best_dinov2l_

## Optional: Pseudo-Labels aus A_equal exportieren

Aktiviere `EXPORT_PSEUDO_LABELS=True` oben falls du V8 mit Pseudo-Labels retrainen willst.

In [6]:
if EXPORT_PSEUDO_LABELS and last_final_probs is not None:
    max_probs = last_final_probs.max(axis=1)
    argmax = last_final_probs.argmax(axis=1)

    selected_idx = []
    print(f"Class-balanced Top-{PSEUDO_TOP_K_PER_CLASS} pro Klasse "
          f"(min conf {PSEUDO_MIN_CONFIDENCE}):")
    for c in range(NUM_CLASSES):
        cand_mask = (argmax == c) & (max_probs >= PSEUDO_MIN_CONFIDENCE)
        cand_idx = np.where(cand_mask)[0]
        cand_idx = cand_idx[np.argsort(-max_probs[cand_idx])]
        chosen = cand_idx[:PSEUDO_TOP_K_PER_CLASS]
        selected_idx.extend(chosen.tolist())
        if len(chosen) > 0:
            print(f"  {c} - {CLASS_NAMES[c]:<22} "
                  f"{len(chosen):>4} ausgewaehlt "
                  f"(min conf={max_probs[chosen].min():.3f}, "
                  f"max conf={max_probs[chosen].max():.3f})")
        else:
            print(f"  {c} - {CLASS_NAMES[c]:<22} KEINE Kandidaten")
    selected_idx = np.array(sorted(selected_idx), dtype=int)
    mask = np.zeros(len(test_df), dtype=bool)
    mask[selected_idx] = True

    pseudo_df = test_df[mask].copy()
    pseudo_df["class_id"] = argmax[mask].astype(int)
    pseudo_df["confidence"] = max_probs[mask]
    pseudo_out = f"pseudo_labels_{TAG.lower()}_v7.csv"
    pseudo_df.to_csv(pseudo_out, index=False)
    print(f"\nPseudo-Labels: {pseudo_out} ({mask.sum()}/{len(test_df)})")
else:
    print("Pseudo-Label Export deaktiviert (EXPORT_PSEUDO_LABELS=False).")

Class-balanced Top-200 pro Klasse (min conf 0.6):
  0 - Lateral_lying_left      200 ausgewaehlt (min conf=0.937, max conf=0.942)
  1 - Lateral_lying_right     200 ausgewaehlt (min conf=0.936, max conf=0.941)
  2 - Sitting                 200 ausgewaehlt (min conf=0.951, max conf=0.993)
  3 - Standing                200 ausgewaehlt (min conf=0.816, max conf=0.822)
  4 - Sternal_lying           200 ausgewaehlt (min conf=0.850, max conf=0.865)

Pseudo-Labels: pseudo_labels_t2_v7.csv (1000/11708)
